# 05 — Fine-tune with progressive unfreezing and recovery checkpoints

This notebook trains the independent NVIDIA FastConformer PC backbone, which is punctuation-aware but does not output diacritics. It performs conservative staged adaptation: top three encoder layers, upper half, then the full model. Every 500 optimizer steps, the training state is saved as a Drive-backed recovery checkpoint. If Colab disconnects or the runtime is stopped, run the same training command again: the project detects the latest checkpoint and resumes the unfinished stage rather than starting that stage from zero.

The console is intentionally compact. It prints one stage header, a progress line every 100 steps, a checkpoint confirmation every 500 steps, and one validation summary. NeMo per-sample RNNT diagnostic predictions are suppressed because they are not the final CTC evaluation output.

In [ ]:
from pathlib import Path
import os

# Each notebook may open in a fresh Colab runtime, so mount Drive before any
# path check rather than relying on a previous notebook's session.
from google.colab import drive
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

PROJECT_DIR = DRIVE_ROOT / "quran-fastconformer-colab"
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
import importlib.util
import subprocess
import sys

# A notebook can be opened after a runtime restart. Install project dependencies
# only when a required module is missing, rather than assuming notebook 01 ran.
required_modules = ("datasets", "jiwer", "soundfile", "yaml", "nemo", "pandas", "matplotlib")

missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
needs_numpy_downgrade = False
if not missing_modules:
    import numpy as np
    needs_numpy_downgrade = int(np.__version__.split(".")[0]) >= 2

if missing_modules or needs_numpy_downgrade:
    reason = missing_modules or ["numpy<2 required by the current NeMo audio loader"]
    print("Installing compatible runtime dependencies:", reason)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", "requirements.txt"])
    print("Dependencies updated. Restart the runtime once before launching a NeMo stage.")
else:
    print("Core project dependencies are available.")


In [ ]:
import os
import torch

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
assert torch.cuda.is_available(), "No GPU. Choose Runtime → Change runtime type → T4 GPU or L4 GPU."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Inspect recovery state before training. This cell never changes the manifest.
import json
from pathlib import Path

state_path = Path("artifacts/experiments/fastconformer_pc/nemo/training_state.json")
checkpoints = sorted(Path("artifacts/experiments/fastconformer_pc/nemo/checkpoints").glob("**/last.ckpt"))
if state_path.exists():
    print(json.loads(state_path.read_text(encoding="utf-8")))
elif checkpoints:
    print("Recovery checkpoints found:")
    for path in checkpoints:
        print(" -", path)
else:
    print("No recovery checkpoint yet. The first run will start stage 1 from the pretrained model.")


In [ ]:
# This command is restart-safe. It resumes an unfinished stage from last.ckpt
# when one exists; completed stages are loaded from their stage_model.nemo export.
!python -m src.train --config configs/fastconformer_quran.yaml --manifest artifacts/manifests/experiment_manifest.json


In [ ]:
import json
from pathlib import Path

summary_path = Path("artifacts/experiments/fastconformer_pc/models/training_summary.json")
state_path = Path("artifacts/experiments/fastconformer_pc/nemo/training_state.json")
if summary_path.exists():
    print(json.loads(summary_path.read_text(encoding="utf-8")))
elif state_path.exists():
    print("Training is not complete yet. Current recovery state:")
    print(json.loads(state_path.read_text(encoding="utf-8")))
else:
    print("No completed training summary yet. Run the training cell first.")
